Demo - 2

Visualize data

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

import matplotlib.pyplot as plt
import seaborn as sns

from scipy import stats

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from keras.optimizers import Adam, SGD, RMSprop
from sklearn.metrics import mean_squared_error, r2_score

from tqdm.keras import TqdmCallback
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.callbacks import ReduceLROnPlateau


from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from tensorflow.keras.utils import to_categorical

In [2]:
import pandas as pd

# Load dataset
url = "http://users.jyu.fi/~olkhriye/ties4911/demos/demo1/Automobile_price_data_Raw_set.csv"
data = pd.read_csv(url)

# Display data
data

,symboling,normalized-losses,make,fuel-type,aspiration,num-of-doors,body-style,drive-wheels,engine-location,wheel-base,...,engine-size,fuel-system,bore,stroke,compression-ratio,horsepower,peak-rpm,city-mpg,highway-mpg,price
0,3,NaN,alfa-romero,gas,std,two,convertible,rwd,front,88.6,...,130,mpfi,3.47,2.68,9.0,111.0,5000.0,21,27,13495.0
1,3,NaN,alfa-romero,gas,std,two,convertible,rwd,front,88.6,...,130,mpfi,3.47,2.68,9.0,111.0,5000.0,21,27,16500.0
2,1,NaN,alfa-romero,gas,std,two,hatchback,rwd,front,94.5,...,152,mpfi,2.68,3.47,9.0,154.0,5000.0,19,26,16500.0
3,2,164.0,audi,gas,std,four,sedan,fwd,front,99.8,...,109,mpfi,3.19,3.40,10.0,102.0,5500.0,24,30,13950.0
4,2,164.0,audi,gas,std,four,sedan,4wd,front,99.4,...,136,mpfi,3.19,3.40,8.0,115.0,5500.0,18,22,17450.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
200,-1,95.0,volvo,gas,std,four,sedan,rwd,front,109.1,...,141,mpfi,3.78,3.15,9.5,114.0,5400.0,23,28,16845.0
201,-1,95.0,volvo,gas,turbo,four,sedan,rwd,front,109.1,...,141,mpfi,3.78,3.15,8.7,160.0,5300.0,19,25,19045.0
202,-1,95.0,volvo,gas,std,four,sedan,rwd,front,109.1,...,173,mpfi,3.58,2.87,8.8,134.0,5500.0,18,23,21485.0
203,-1,95.0,volvo,diesel,turbo,four,sedan,rwd,front,109.1,...,145,idi,3.01,3.40,23.0,106.0,4800.0,26,27,22470.0


In [3]:
features = ['make', 'body-style', 'wheel-base', 'engine-size', 'horsepower', 'peak-rpm', 'highway-mpg', 'price']
df = data[features]
df

,make,body-style,wheel-base,engine-size,horsepower,peak-rpm,highway-mpg,price
0,alfa-romero,convertible,88.6,130,111.0,5000.0,27,13495.0
1,alfa-romero,convertible,88.6,130,111.0,5000.0,27,16500.0
2,alfa-romero,hatchback,94.5,152,154.0,5000.0,26,16500.0
3,audi,sedan,99.8,109,102.0,5500.0,30,13950.0
4,audi,sedan,99.4,136,115.0,5500.0,22,17450.0
...,...,...,...,...,...,...,...,...
200,volvo,sedan,109.1,141,114.0,5400.0,28,16845.0
201,volvo,sedan,109.1,141,160.0,5300.0,25,19045.0
202,volvo,sedan,109.1,173,134.0,5500.0,23,21485.0
203,volvo,sedan,109.1,145,106.0,4800.0,27,22470.0


In [54]:
# Split features and target
X = df[["body-style", "wheel-base", "engine-size", "horsepower", "peak-rpm", "highway-mpg"]]
y = df["make"]
y = label_encoder.fit_transform(y)
y = to_categorical(y)

In [57]:
# Preprocessing: One-hot encoding for categorical and scaling for numeric features
categorical_features = ["body-style"]
numeric_features = ["wheel-base", "engine-size", "horsepower", "peak-rpm", "highway-mpg"]

preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_features),
        ("cat", OneHotEncoder(), categorical_features)
    ]
)

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Preprocess data
X_train = preprocessor.fit_transform(X_train)
X_test = preprocessor.transform(X_test)

Training the *model*

In [61]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping
from tqdm.keras import TqdmCallback

# Define the model
model = Sequential([
    Dense(64, activation="relu", input_shape=(X_train.shape[1],)),
    Dropout(0.2),
    Dense(32, activation="relu"),
    Dropout(0.2),
    Dense(23, activation="softmax")  # Output layer for multiclass classification
])

# Compile the model
model.compile(optimizer=Adam(learning_rate=0.001),
              loss="categorical_crossentropy",
              metrics=["accuracy"])

# Model summary
model.summary()

# Early stopping callback
early_stopping = EarlyStopping(monitor="val_loss", patience=10, restore_best_weights=True)

# Train the model
history = model.fit(X_train, y_train,
                    epochs=100,
                    batch_size=16,
                    validation_data=(X_test, y_test),
                    callbacks=[TqdmCallback(verbose=1)],
                    verbose=0)

# Evaluate the model
test_loss, test_acc = model.evaluate(X_test, y_test, verbose=0)
print(f"Test Accuracy: {test_acc:.4f}")
print(f"Test Loss: {test_loss:.4f}")

# Plot training history
plt.figure(figsize=(12, 5))

# Plot accuracy
plt.subplot(1, 2, 1)
plt.plot(history.history["accuracy"], label="Training Accuracy")
plt.plot(history.history["val_accuracy"], label="Validation Accuracy")
plt.title("Accuracy Over Epochs")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.legend()

# Plot loss
plt.subplot(1, 2, 2)
plt.plot(history.history["loss"], label="Training Loss")
plt.plot(history.history["val_loss"], label="Validation Loss")
plt.title("Loss Over Epochs")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()

plt.tight_layout()
plt.show()

/usr/local/lib/python3.11/dist-packages/keras/src/layers/core/dense.py:87: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential_4"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ dense_16 (Dense)                     │ (None, 64)                  │             704 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_12 (Dropout)                 │ (None, 64)                  │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_17 (Dense)                     │ (None, 32)                  │           2,080 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_13 (Dropout)                 │ (None, 32)                  │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_18 (Dense)                     │ (None, 23)                  │             759 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 3,543 (13.84 KB)

 Trainable params: 3,543 (13.84 KB)

 Non-trainable params: 0 (0.00 B)

0epoch [00:00, ?epoch/s]

0batch [00:00, ?batch/s]

ValueError: Arguments `target` and `output` must have the same shape. Received: target.shape=(None, 21), output.shape=(None, 23)

TEST

In [67]:
encoded_body_style_df = pd.DataFrame(encoded_body_style, columns=encoder.get_feature_names_out(['body-style']))

ValueError: input_features is not equal to feature_names_in_

In [65]:
pd.DataFrame(encoded_body_style, columns=encoder.get_feature_names_out(['body-style']))

ValueError: input_features is not equal to feature_names_in_

In [68]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense

# Load the dataset
url = 'http://users.jyu.fi/~olkhriye/ties4911/demos/demo1/Automobile_price_data_Raw_set.csv'
df = pd.read_csv(url)

# Select relevant columns
columns = ['make', 'body-style', 'wheel-base', 'engine-size', 'horsepower', 'peak-rpm', 'highway-mpg', 'price']
df = df[columns]

# Handle missing values
df.dropna(inplace=True)

# Encode categorical variables
# Create separate encoder objects for each categorical feature
encoder_body_style = OneHotEncoder(sparse_output=False)  # Encoder for body-style
encoder_make = OneHotEncoder(sparse_output=False)  # Encoder for make

encoded_body_style = encoder_body_style.fit_transform(df[['body-style']]) # Use specific encoder
encoded_make = encoder_make.fit_transform(df[['make']]) # Use specific encoder


# Create DataFrame for encoded features
encoded_body_style_df = pd.DataFrame(encoded_body_style, columns=encoder_body_style.get_feature_names_out(['body-style'])) # Use specific encoder
encoded_make_df = pd.DataFrame(encoded_make, columns=encoder_make.get_feature_names_out(['make'])) # Use specific encoder

# Concatenate encoded features with numerical features
numerical_features = df[['wheel-base', 'engine-size', 'horsepower', 'peak-rpm', 'highway-mpg', 'price']]
X = pd.concat([numerical_features.reset_index(drop=True), encoded_body_style_df.reset_index(drop=True)], axis=1)
y = encoded_make_df

# Normalize numerical features
scaler = StandardScaler()
X[numerical_features.columns] = scaler.fit_transform(numerical_features)

# Split the data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# Build the neural network model
model = Sequential()
model.add(Dense(64, input_dim=X_train.shape[1], activation='relu'))
model.add(Dense(32, activation='relu'))
model.add(Dense(y_train.shape[1], activation='softmax'))

# Compile the model
model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])

# Train the model
model.fit(X_train, y_train, epochs=100, batch_size=10, validation_data=(X_test, y_test))

# Example predictions
import numpy as np

# Function to preprocess input data
def preprocess_input(input_data):
    input_df = pd.DataFrame([input_data], columns=['body-style', 'wheel-base', 'engine-size', 'horsepower', 'peak-rpm', 'highway-mpg', 'price'])
    input_df['body-style'] = encoder_body_style.transform(input_df[['body-style']]) # Use specific encoder
    input_df[numerical_features.columns] = scaler.transform(input_df[numerical_features.columns])
    return input_df

/usr/local/lib/python3.11/dist-packages/keras/src/layers/core/dense.py:87: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Epoch 1/100
14/14 ━━━━━━━━━━━━━━━━━━━━ 4s 121ms/step - accuracy: 0.0603 - loss: 2.9524 - val_accuracy: 0.1017 - val_loss: 2.9834
Epoch 2/100
14/14 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.1529 - loss: 2.8915 - val_accuracy: 0.1695 - val_loss: 2.9118
Epoch 3/100
14/14 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.2946 - loss: 2.7396 - val_accuracy: 0.1525 - val_loss: 2.8553
Epoch 4/100
14/14 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.2640 - loss: 2.6735 - val_accuracy: 0.1525 - val_loss: 2.7887
Epoch 5/100
14/14 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.2640 - loss: 2.5365 - val_accuracy: 0.1525 - val_loss: 2.7420
Epoch 6/100
14/14 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.2963 - loss: 2.4266 - val_accuracy: 0.1864 - val_loss: 2.6759
Epoch 7/100
14/14 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.2397 - loss: 2.3595 - val_accuracy: 0.1695 - val_loss: 2.6188
Epoch 8/100
14/14 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.2871 - loss: 2.1796 - val_accuracy: 0.1695 -

ValueError: could not broadcast input array from shape (5,1) into shape (1,1)

In [70]:
X_train.shape

(135, 11)

In [69]:
# Input data
input_1 = ['sedan', 103.5, 164, 121, 4250, 25, 24565]
input_2 = ['hatchback', 86.6, 92, 58, 4800, 54, 6479]

# Preprocess inputs
processed_input_1 = preprocess_input(input_1)
processed_input_2 = preprocess_input(input_2)

# Predict
prediction_1 = model.predict(processed_input_1)
prediction_2 = model.predict(processed_input_2)

# Decode predictions
predicted_make_1 = encoder_make.inverse_transform(prediction_1) # Use specific encoder
predicted_make_2 = encoder_make.inverse_transform(prediction_2) # Use specific encoder

print(f"Predicted make for input 1: {predicted_make_1[0]}")
print(f"Predicted make for input 2: {predicted_make_2[0]}")

ValueError: could not broadcast input array from shape (5,1) into shape (1,1)